# Pure Hugging Face Strong Teacher for ATE Pseudo-labeling

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import re
import ast
import json
import random
from pathlib import Path
from collections import Counter

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    set_seed,
)

set_seed(42)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


## 3. Load gold dataset

In [2]:
SHEET_ID = '1KXcA0PPOpygla1inEfnTc10FND6DFNL8p8vpQKAX6Tw'
gold_csv_url = f'https://docs.google.com/spreadsheets/d/{SHEET_ID}/export?format=csv'

gold_local_path = ""

sentence_col = "sentence_text"
triplets_col = "triplets"

def load_csv(csv_url="", local_path="", upload_prompt="Upload CSV file"):
    if csv_url.strip():
        return pd.read_csv(csv_url)

    if local_path.strip():
        return pd.read_csv(local_path)

    try:
        from google.colab import files
        print(upload_prompt)
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No file uploaded.")
        path = next(iter(uploaded.keys()))
        return pd.read_csv(path)
    except Exception as e:
        raise RuntimeError("Set csv_url/local_path, or upload a CSV file.") from e

gold_df = load_csv(
    gold_csv_url,
    gold_local_path,
    upload_prompt="Upload GOLD CSV file"
)

missing = {sentence_col, triplets_col} - set(gold_df.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}. Available columns: {list(gold_df.columns)}")

print("Gold shape:", gold_df.shape)
display(gold_df.head())

Gold shape: (4000, 6)


,parent_asin,sentence_id,sentence_text,rating,triplets,category_name
0,B07DGRVTWF,3,basically a poor implementation of the alexa p...,2.0,"[[""alexa platform"", ""poor implementation"", 0]]",electronics_p2
1,B07BFPJ6VX,4,support through direct messaging was great no ...,5.0,"[[""support through direct messaging"", ""great"",...",electronics_p2
2,B0B1PYNH8X,1,for echo show 5 i have the 2nd generation echo...,5.0,[],electronics_p2
3,B0007X9JMA,4,br br note i ve only used it for regular compu...,5.0,[],electronics_p2
4,B00N63V39K,6,wanted to use for business but i m in sales an...,1.0,[],electronics_p2


## 4. Parse triplets

In [3]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def safe_parse_triplets(value):
    if isinstance(value, list):
        return value

    if pd.isna(value):
        return []

    text = str(value).strip()

    if not text or text.lower() in {"nan", "none", "null"}:
        return []

    for candidate in [text, text.replace('""', '"')]:
        try:
            parsed = ast.literal_eval(candidate)
            if isinstance(parsed, list):
                return parsed
        except Exception:
            pass

    return []

def extract_aspects(triplets):
    aspects = []

    for item in triplets:
        if not isinstance(item, (list, tuple)) or len(item) == 0:
            continue

        aspect = clean_text(item[0])
        if aspect:
            aspects.append(aspect)

    seen = set()
    unique = []

    for aspect in aspects:
        if aspect not in seen:
            seen.add(aspect)
            unique.append(aspect)

    return unique

gold_df = gold_df.copy()
gold_df[sentence_col] = gold_df[sentence_col].apply(clean_text)
gold_df["triplets_parsed"] = gold_df[triplets_col].apply(safe_parse_triplets)
gold_df["gold_aspects"] = gold_df["triplets_parsed"].apply(extract_aspects)
gold_df["has_aspect"] = gold_df["gold_aspects"].apply(lambda x: len(x) > 0)

gold_df = gold_df[gold_df[sentence_col].str.len() > 0].reset_index(drop=True)

print("Rows:", len(gold_df))
print("Has aspect:", int(gold_df["has_aspect"].sum()))
print("No aspect:", int((~gold_df["has_aspect"]).sum()))
print("Has-aspect ratio:", round(gold_df["has_aspect"].mean(), 4))

display(gold_df[[sentence_col, "gold_aspects", "has_aspect"]].head(10))

Rows: 4000
Has aspect: 2271
No aspect: 1729
Has-aspect ratio: 0.5678


,sentence_text,gold_aspects,has_aspect
0,basically a poor implementation of the alexa p...,[alexa platform],True
1,support through direct messaging was great no ...,[support through direct messaging],True
2,for echo show 5 i have the 2nd generation echo...,[],False
3,br br note i ve only used it for regular compu...,[],False
4,wanted to use for business but i m in sales an...,[],False
5,i use it everyday to backup important files,[],False
6,as well as occasional book reading and online ...,[],False
7,the other major con is that it does not shut d...,[],False
8,i ve bought other screen protectors thatsay go...,[],False
9,i like this ipad cover i like this ipad cover,[ipad cover],True


## 5. Gold split

In [4]:
train_df, test_df = train_test_split(
    gold_df,
    test_size=0.20,
    random_state=42,
    shuffle=True,
    stratify=gold_df["has_aspect"] if gold_df["has_aspect"].nunique() == 2 else None,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

for name, part in [("train", train_df), ("test", test_df)]:
    print(
        name,
        part.shape,
        "has_aspect:",
        int(part["has_aspect"].sum()),
        "no_aspect:",
        int((~part["has_aspect"]).sum()),
        "ratio:",
        round(part["has_aspect"].mean(), 4),
    )

train (3200, 9) has_aspect: 1817 no_aspect: 1383 ratio: 0.5678
test (800, 9) has_aspect: 454 no_aspect: 346 ratio: 0.5675


In [7]:
train_df.to_csv("gold_train.csv", index=False)
test_df.to_csv("gold_test.csv", index=False)

# Part A — Pure HF Gate Teacher

In [9]:

# gate_model_name = "roberta-large"
gate_model_name = "roberta-base"

gate_tokenizer = AutoTokenizer.from_pretrained(gate_model_name, use_fast=True)

class GateDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        enc = self.tokenizer(
            row[sentence_col],
            truncation=True,
            max_length=self.max_length,
        )
        enc["labels"] = int(row["has_aspect"])
        return enc

gate_train_dataset = GateDataset(train_df, gate_tokenizer, max_length=192)
gate_test_dataset = GateDataset(test_df, gate_tokenizer, max_length=192)

print("Gate train samples:", len(gate_train_dataset))
print("Gate test samples :", len(gate_test_dataset))

Gate train samples: 3200
Gate test samples : 800


In [15]:
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
import torch
import os
import json


gate_model = AutoModelForSequenceClassification.from_pretrained(
    gate_model_name,
    num_labels=2,
    id2label={0: "no_aspect", 1: "has_aspect"},
    label2id={"no_aspect": 0, "has_aspect": 1},
).to(device)


def make_training_args(
    output_dir,
    lr=2e-5,
    epochs=3,
    train_bs=4,
    eval_bs=8,
    grad_accum=4,
):
    common_kwargs = dict(
        output_dir=output_dir,
        seed=42,

        do_eval=False,

        num_train_epochs=epochs,
        learning_rate=lr,
        weight_decay=0.01,
        warmup_ratio=0.1,
        per_device_train_batch_size=train_bs,
        per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=grad_accum,
        fp16=torch.cuda.is_available(),

        logging_strategy="steps",
        logging_steps=50,
        save_strategy="epoch",
        save_total_limit=2,
        report_to="none",

        load_best_model_at_end=False,
    )

    try:
        return TrainingArguments(
            eval_strategy="no",
            **common_kwargs,
        )
    except TypeError:
        return TrainingArguments(
            evaluation_strategy="no",
            **common_kwargs,
        )


class BestTrainLossTrainer(Trainer):
    def __init__(self, *args, best_model_dir=None, tokenizer_to_save=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.best_train_loss = float("inf")
        self.best_train_step = None
        self.best_model_dir = best_model_dir
        self.tokenizer_to_save = tokenizer_to_save

    def log(self, logs, *args, **kwargs):
        super().log(logs, *args, **kwargs)

        if "loss" not in logs:
            return

        train_loss = float(logs["loss"])

        if train_loss < self.best_train_loss:
            self.best_train_loss = train_loss
            self.best_train_step = self.state.global_step

            if self.best_model_dir is not None:
                os.makedirs(self.best_model_dir, exist_ok=True)

                self.save_model(self.best_model_dir)

                if self.tokenizer_to_save is not None:
                    self.tokenizer_to_save.save_pretrained(self.best_model_dir)

                with open(os.path.join(self.best_model_dir, "best_train_loss.json"), "w") as f:
                    json.dump(
                        {
                            "best_train_loss": self.best_train_loss,
                            "best_train_step": self.best_train_step,
                        },
                        f,
                        indent=2,
                    )

                print(
                    f"\nNew best train loss: {self.best_train_loss:.6f} "
                    f"at step {self.best_train_step}. "
                    f"Saved to {self.best_model_dir}"
                )


gate_args = make_training_args(
    output_dir="/content/hf_gate_teacher",
    lr=1e-5,
    epochs=5,
    train_bs=4,
    eval_bs=16,
    grad_accum=4,
)


gate_trainer = BestTrainLossTrainer(
    model=gate_model,
    args=gate_args,
    train_dataset=gate_train_dataset,
    processing_class=gate_tokenizer,
    best_model_dir="/content/drive/MyDrive/ate_gate_teacher_phase1/",
    tokenizer_to_save=gate_tokenizer,
)


gate_trainer.train()

print("Best train loss:", gate_trainer.best_train_loss)
print("Best train step:", gate_trainer.best_train_step)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,2.733094
100,2.663726
150,2.390879
200,2.064306
250,1.882709
300,1.684385
350,1.723171
400,1.697848
450,1.229289
500,1.295397


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


New best train loss: 2.733094 at step 50. Saved to /content/drive/MyDrive/ate_gate_teacher_phase1/


...


New best train loss: 0.807215 at step 850. Saved to /content/drive/MyDrive/ate_gate_teacher_phase1/


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


New best train loss: 0.712099 at step 950. Saved to /content/drive/MyDrive/ate_gate_teacher_phase1/


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Best train loss: 0.7120994567871094
Best train step: 950


## 8. Evaluate gate teacher on test

In [16]:
gate_model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/ate_gate_teacher_phase1/").to(device)
gate_model.eval()

def predict_gate_proba(sentences, batch_size=32):
    probs_all = []

    for i in tqdm(range(0, len(sentences), batch_size), desc="Gate predicting"):
        batch = sentences[i:i+batch_size]
        enc = gate_tokenizer(
            batch,
            truncation=True,
            max_length=192,
            padding=True,
            return_tensors="pt",
        ).to(device)

        with torch.no_grad():
            logits = gate_model(**enc).logits
            probs = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()

        probs_all.extend(probs.tolist())

    return np.array(probs_all)

X_gate_test = test_df[sentence_col].tolist()
y_gate_test = test_df["has_aspect"].astype(int).to_numpy()

gate_test_proba = predict_gate_proba(X_gate_test, batch_size=32)
gate_test_pred = (gate_test_proba >= 0.5).astype(int)

print("Accuracy:", round(accuracy_score(y_gate_test, gate_test_pred), 4))
print()
print(classification_report(
    y_gate_test,
    gate_test_pred,
    target_names=["no_aspect", "has_aspect"],
    digits=4,
    zero_division=0,
))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_gate_test, gate_test_pred))

try:
    print("ROC-AUC:", round(roc_auc_score(y_gate_test, gate_test_proba), 4))
except Exception as e:
    print("ROC-AUC unavailable:", e)

threshold_rows = []

for threshold in [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    pred = (gate_test_proba >= threshold).astype(int)

    p, r, f1, _ = precision_recall_fscore_support(
        y_gate_test,
        pred,
        labels=[1],
        average="binary",
        zero_division=0,
    )

    cm = confusion_matrix(y_gate_test, pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()

    threshold_rows.append({
        "threshold": threshold,
        "has_aspect_precision": p,
        "has_aspect_recall": r,
        "has_aspect_f1": f1,
        "no_aspect_fpr": fp / (fp + tn) if (fp + tn) else 0.0,
        "pass_rate": pred.mean(),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    })

gate_threshold_df = pd.DataFrame(threshold_rows)
display(gate_threshold_df)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Gate predicting:   0%|          | 0/25 [00:00<?, ?it/s]

Accuracy: 0.835

              precision    recall  f1-score   support

   no_aspect     0.8690    0.7283    0.7925       346
  has_aspect     0.8157    0.9163    0.8631       454

    accuracy                         0.8350       800
   macro avg     0.8423    0.8223    0.8278       800
weighted avg     0.8387    0.8350    0.8325       800

Confusion matrix [[TN, FP], [FN, TP]]:
[[252  94]
 [ 38 416]]
ROC-AUC: 0.9152


,threshold,has_aspect_precision,has_aspect_recall,has_aspect_f1,no_aspect_fpr,pass_rate,tn,fp,fn,tp
0,0.10,0.768142,0.955947,0.851816,0.378613,0.70625,215,131,20,434
1,0.20,0.798131,0.940529,0.863498,0.312139,0.66875,238,108,27,427
2,0.30,0.801136,0.931718,0.861507,0.303468,0.66000,241,105,31,423
3,0.40,0.803435,0.927313,0.860941,0.297688,0.65500,243,103,33,421
4,0.50,0.815686,0.916300,0.863071,0.271676,0.63750,252,94,38,416
5,0.60,0.821643,0.903084,0.860441,0.257225,0.62375,257,89,44,410
6,0.70,0.835052,0.892070,0.862620,0.231214,0.60625,266,80,49,405
7,0.80,0.854664,0.867841,0.861202,0.193642,0.57625,279,67,60,394
8,0.90,0.877358,0.819383,0.847380,0.150289,0.53000,294,52,82,372
9,0.95,0.894872,0.768722,0.827014,0.118497,0.48750,305,41,105,349


# Part B — Pure HF ATE Teacher

In [17]:
TOKEN_RE = re.compile(r"\b\w+(?:'\w+)?\b")

label_list = ["O", "B-ASP", "I-ASP"]
label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for label, i in label2id.items()}

def tokenize_words(text):
    return [m.group(0) for m in TOKEN_RE.finditer(text)]

def normalize_aspect(text):
    text = clean_text(str(text))
    text = re.sub(r"[^a-z0-9\s]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def aspect_to_tokens(aspect):
    normalized = normalize_aspect(aspect)
    if not normalized:
        return []
    return tokenize_words(normalized)

def create_bio_labels(sentence, aspects):
    tokens = tokenize_words(sentence)
    labels = ["O"] * len(tokens)

    token_norm = [normalize_aspect(tok) for tok in tokens]
    unmatched = []

    for aspect in aspects:
        asp_tokens = aspect_to_tokens(aspect)

        if not asp_tokens:
            continue

        n = len(asp_tokens)
        found = False

        for i in range(0, len(tokens) - n + 1):
            if token_norm[i:i+n] == asp_tokens:
                labels[i] = "B-ASP"
                for j in range(i + 1, i + n):
                    labels[j] = "I-ASP"
                found = True

        if not found:
            unmatched.append(aspect)

    return tokens, labels, unmatched

def add_ate_columns(dataframe, desc):
    rows = []
    n_gold_aspects = 0
    n_unmatched = 0
    unmatched_counter = Counter()

    for idx, row in tqdm(dataframe.iterrows(), total=len(dataframe), desc=desc):
        tokens, labels, unmatched = create_bio_labels(row[sentence_col], row["gold_aspects"])

        n_gold_aspects += len(row["gold_aspects"])
        n_unmatched += len(unmatched)

        for aspect in unmatched:
            unmatched_counter[aspect] += 1

        rows.append({
            "id": idx,
            "sentence": row[sentence_col],
            "tokens": tokens,
            "word_labels": labels,
            "gold_aspects": row["gold_aspects"],
            "has_aspect": bool(row["has_aspect"]),
            "unmatched_aspects": unmatched,
        })

    out = pd.DataFrame(rows)
    match_rate = 1 - n_unmatched / max(n_gold_aspects, 1)

    return out, {
        "n_gold_aspects": n_gold_aspects,
        "n_unmatched": n_unmatched,
        "match_rate": match_rate,
        "unmatched_counter": unmatched_counter,
    }

ate_train_source = train_df[train_df["has_aspect"]].copy().reset_index(drop=True)
ate_test_source = test_df[test_df["has_aspect"]].copy().reset_index(drop=True)

ate_train_df, ate_train_info = add_ate_columns(ate_train_source, "BIO train")
ate_test_df, ate_test_info = add_ate_columns(ate_test_source, "BIO test")

print("ATE train:", ate_train_df.shape, {k: v for k, v in ate_train_info.items() if k != "unmatched_counter"})
print("ATE test :", ate_test_df.shape, {k: v for k, v in ate_test_info.items() if k != "unmatched_counter"})
print("Top train unmatched:", ate_train_info["unmatched_counter"].most_common(20))
print("Top test unmatched :", ate_test_info["unmatched_counter"].most_common(20))

BIO train:   0%|          | 0/1817 [00:00<?, ?it/s]

BIO test:   0%|          | 0/454 [00:00<?, ?it/s]

ATE train: (1817, 7) {'n_gold_aspects': 2564, 'n_unmatched': 91, 'match_rate': 0.9645085803432137}
ATE test : (454, 7) {'n_gold_aspects': 656, 'n_unmatched': 24, 'match_rate': 0.9634146341463414}
Top train unmatched: [('story', 4), ('it', 4), ('value', 4), ('price', 4), ('book', 4), ('compatibility', 3), ('fit', 3), ('writing', 3), ('connection to my phone', 2), ('ending', 2), ('quality', 2), ('functionality', 2), ('device', 1), ('character', 1), ("gabaldon's command", 1), ('blue ink cartridges', 1), ("nora's trilogy sets", 1), ("maggie thom's captured lies", 1), ('accuracy', 1), ('syncing with my iphone and strava', 1)]
Top test unmatched : [('game', 2), ('it', 2), ('installation', 2), ('nba game app', 1), ('volume', 1), ('usb hub', 1), ('fit', 1), ('value', 1), ('situation', 1), ('delivery', 1), ('advertisement', 1), ('conducción', 1), ('conference calling', 1), ('compatibility with apple and windows', 1), ('assembly', 1), ('packaging', 1), ('iPad', 1), ('labeling', 1), ('brightness'

In [18]:
for name, part in [("train", ate_train_df), ("test", ate_test_df)]:
    c = Counter()
    for labels in part["word_labels"]:
        c.update(labels)
    print(name, c)

    if name == "train" and c["B-ASP"] == 0:
        raise ValueError("No B-ASP labels in train. Check aspect matching.")

display(
    ate_train_df[ate_train_df["word_labels"].apply(lambda xs: "B-ASP" in xs)][
        ["sentence", "tokens", "word_labels", "gold_aspects"]
    ].head(10)
)

train Counter({'O': 25491, 'B-ASP': 2701, 'I-ASP': 972})
test Counter({'O': 6385, 'B-ASP': 678, 'I-ASP': 254})


,sentence,tokens,word_labels,gold_aspects
1,i put in dark knight on blu ray and any of my ...,"[i, put, in, dark, knight, on, blu, ray, and, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[picture, colors, blacks]"
2,the sound is amazing and connected to my phone...,"[the, sound, is, amazing, and, connected, to, ...","[O, B-ASP, O, O, O, O, O, O, O, O, O]","[sound, connection to my phone]"
4,the chalk pen works well and washes off easily...,"[the, chalk, pen, works, well, and, washes, of...","[O, B-ASP, I-ASP, O, O, O, O, O, O, O, O, O, O]",[chalk pen]
5,gabaldon's command of our language is commenda...,"[gabaldon's, command, of, our, language, is, c...","[O, O, O, O, O, O, O, O, O, B-ASP, O, O, B-ASP]","[gabaldon's command, scenes, emotions]"
6,five [GENERIC_NOUN] great price fans are quit ...,"[five, GENERIC_NOUN, great, price, fans, are, ...","[O, O, O, O, B-ASP, O, O, O, O, O, O]",[fans]
8,even better that my kindle copy was free,"[even, better, that, my, kindle, copy, was, free]","[O, O, O, O, B-ASP, I-ASP, O, O]",[kindle copy]
9,did not even rate but was made to the submit t...,"[did, not, even, rate, but, was, made, to, the...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...",[picture]
10,the wireless signal did not reach as far as th...,"[the, wireless, signal, did, not, reach, as, f...","[O, B-ASP, I-ASP, O, O, O, O, O, O, O, O, O]",[wireless signal]
11,loved the characters and the plot,"[loved, the, characters, and, the, plot]","[O, O, B-ASP, O, O, B-ASP]","[characters, plot]"
13,the best feature of this program is the abilit...,"[the, best, feature, of, this, program, is, th...","[O, O, B-ASP, O, O, O, O, O, O, O, O, O, O, O,...",[feature]


In [19]:
# ate_model_name = "roberta-large"
ate_model_name = "roberta-base"

ate_tokenizer = AutoTokenizer.from_pretrained(ate_model_name, use_fast=True)

class ATEDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=192):
        self.dataframe = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        tokens = row["tokens"]
        word_labels = row["word_labels"]
        word_label_ids = [label2id[x] for x in word_labels]

        enc = self.tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            max_length=self.max_length,
        )

        labels = []
        previous_word_id = None

        for word_id in enc.word_ids():
            if word_id is None:
                labels.append(-100)
            elif word_id != previous_word_id:
                labels.append(word_label_ids[word_id])
            else:
                labels.append(-100)

            previous_word_id = word_id

        enc["labels"] = labels
        return enc

ate_train_dataset = ATEDataset(ate_train_df, ate_tokenizer, max_length=192)
ate_test_dataset = ATEDataset(ate_test_df, ate_tokenizer, max_length=192)

data_collator = DataCollatorForTokenClassification(tokenizer=ate_tokenizer)

print("ATE train samples:", len(ate_train_dataset))
print("ATE test samples :", len(ate_test_dataset))

ATE train samples: 1817
ATE test samples : 454


## 12. Weighted loss Trainer

In [24]:
from transformers import TrainerCallback


class SaveBestTrainLossCallback(TrainerCallback):
    def __init__(self, save_dir, tokenizer=None):
        self.save_dir = save_dir
        self.tokenizer = tokenizer
        self.best_loss = float("inf")

    def on_log(self, args, state, control, logs=None, model=None, **kwargs):
        if logs is None or "loss" not in logs:
            return control

        loss = logs["loss"]

        if loss < self.best_loss:
            self.best_loss = loss

            if state.is_world_process_zero:
                os.makedirs(self.save_dir, exist_ok=True)

                model_to_save = getattr(model, "module", model)
                model_to_save.save_pretrained(self.save_dir)

                if self.tokenizer is not None:
                    self.tokenizer.save_pretrained(self.save_dir)

                with open(os.path.join(self.save_dir, "best_train_loss.json"), "w") as f:
                    json.dump(
                        {
                            "best_train_loss": float(loss),
                            "global_step": int(state.global_step),
                            "epoch": float(state.epoch) if state.epoch is not None else None,
                        },
                        f,
                        indent=2,
                    )

                print(
                    f"\nSaved best train loss model: "
                    f"loss={loss:.6f}, step={state.global_step}, epoch={state.epoch}"
                )

        return control

In [21]:
label_counts = Counter()
for labels in ate_train_df["word_labels"]:
    label_counts.update(labels)

counts = torch.tensor([label_counts[label] for label in label_list], dtype=torch.float)
freq = counts / counts.sum()

class_weights = 1.0 / torch.sqrt(freq)
class_weights = class_weights / class_weights.mean()
class_weights = torch.clamp(class_weights, min=0.25, max=5.0)

print("ATE label counts:", label_counts)
print("Class weights:")
for label, weight in zip(label_list, class_weights.tolist()):
    print(label, round(weight, 4))

class WeightedTokenTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        loss_fct = nn.CrossEntropyLoss(
            weight=self.class_weights.to(logits.device) if self.class_weights is not None else None,
            ignore_index=-100,
        )

        loss = loss_fct(
            logits.view(-1, model.config.num_labels),
            labels.view(-1),
        )

        return (loss, outputs) if return_outputs else loss

ATE label counts: Counter({'O': 25491, 'B-ASP': 2701, 'I-ASP': 972})
Class weights:
O 0.3263
B-ASP 1.0025
I-ASP 1.6712


In [30]:
ate_model = AutoModelForTokenClassification.from_pretrained(
    ate_model_name,
    num_labels=len(label_list),
    id2label=id2label,
    label2id=label2id,
).to(device)

if hasattr(ate_model, "gradient_checkpointing_enable"):
    ate_model.gradient_checkpointing_enable()

ate_args = make_training_args(
    output_dir="/content/ate_teacher_phase1/",
    lr=1e-5,
    epochs=5,
    train_bs=2,
    eval_bs=8,
    grad_accum=8,
)

best_train_loss_dir = "/content/drive/MyDrive/ate_teacher_phase1/"

ate_trainer = WeightedTokenTrainer(
    class_weights=class_weights,
    model=ate_model,
    args=ate_args,
    train_dataset=ate_train_dataset,
    processing_class=ate_tokenizer,
    data_collator=data_collator,
    callbacks=[
        SaveBestTrainLossCallback(
            save_dir=best_train_loss_dir,
            tokenizer=ate_tokenizer,
        )
    ],
)

ate_trainer.train()

ate_trainer.save_model("/content/hf_ate_teacher/final")
ate_tokenizer.save_pretrained("/content/hf_ate_teacher/final")

print("Saved ATE teacher to /content/hf_ate_teacher/final")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForTokenClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
classifier.bias                 | MISSING    | 
classifier.weight               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Step,Training Loss
50,8.505162
100,4.502151
150,3.347839
200,2.962887
250,2.784496
300,2.523604
350,2.323932
400,2.179539
450,2.128786
500,2.197740


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved best train loss model: loss=8.505162, step=50, epoch=0.44004400440044006


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Saved best train loss model: loss=2.061303, step=550, epoch=4.827282728272827


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved ATE teacher to /content/hf_ate_teacher/final


## 14. ATE evaluation on test

In [33]:
ate_model = AutoModelForTokenClassification.from_pretrained("/content/hf_ate_teacher/best_train_loss").to(device)
ate_model.eval()

def normalize_span(span):
    span = clean_text(span)
    span = re.sub(r"\s+", " ", span)
    return span.strip(" \t\n\r.,;:!?()[]{}\"'")

def labels_to_spans(tokens, labels, confs=None):
    spans = []
    i = 0

    while i < len(labels):
        if labels[i] == "B-ASP":
            start = i
            i += 1

            while i < len(labels) and labels[i] == "I-ASP":
                i += 1

            end = i - 1
            text = normalize_span(" ".join(tokens[start:end+1]))

            if text:
                if confs is not None:
                    span_conf = float(np.mean(confs[start:end+1]))
                else:
                    span_conf = 1.0

                spans.append({
                    "start": start,
                    "end": end,
                    "text": text,
                    "conf": span_conf,
                })
        else:
            i += 1

    return spans

def gold_span_set(row):
    return {x["text"] for x in labels_to_spans(row["tokens"], row["word_labels"])}

def predict_ate_for_tokens(tokens):
    enc = ate_tokenizer(
        tokens,
        is_split_into_words=True,
        truncation=True,
        max_length=192,
        return_tensors="pt",
    )

    word_ids = enc.word_ids()
    enc = {k: v.to(device) for k, v in enc.items()}

    with torch.no_grad():
        logits = ate_model(**enc).logits[0]
        probs = torch.softmax(logits, dim=-1).detach().cpu().numpy()
        pred_ids = probs.argmax(axis=-1)

    word_labels = ["O"] * len(tokens)
    word_confs = [0.0] * len(tokens)

    seen = set()

    for tok_idx, word_id in enumerate(word_ids):
        if word_id is None or word_id in seen:
            continue

        seen.add(word_id)

        pred_id = int(pred_ids[tok_idx])
        word_labels[word_id] = id2label[pred_id]
        word_confs[word_id] = float(probs[tok_idx, pred_id])

    spans = labels_to_spans(tokens, word_labels, word_confs)

    return {
        "tokens": tokens,
        "labels": word_labels,
        "confs": word_confs,
        "spans": spans,
    }

def evaluate_span_rows(rows):
    tp = fp = fn = 0

    for item in rows:
        gold = item["gold"]
        pred = item["pred"]

        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

thresholds = [0.0, 0.5, 0.7, 0.8, 0.9, 0.95]
threshold_results = []

all_test_predictions = []

for _, row in tqdm(ate_test_df.iterrows(), total=len(ate_test_df), desc="ATE test predicting"):
    pred = predict_ate_for_tokens(row["tokens"])
    all_test_predictions.append({
        "row": row,
        "pred": pred,
    })

for threshold in thresholds:
    rows = []

    for item in all_test_predictions:
        row = item["row"]
        pred = item["pred"]

        pred_set = {s["text"] for s in pred["spans"] if s["conf"] >= threshold}

        rows.append({
            "gold": gold_span_set(row),
            "pred": pred_set,
        })

    m = evaluate_span_rows(rows)
    m["threshold"] = threshold
    threshold_results.append(m)

ate_threshold_df = pd.DataFrame(threshold_results)
display(ate_threshold_df[[
    "threshold", "precision", "recall", "f1", "tp", "fp", "fn"
]])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

ATE test predicting:   0%|          | 0/454 [00:00<?, ?it/s]

,threshold,precision,recall,f1,tp,fp,fn
0,0.00,0.532067,0.708861,0.607870,448,394,184
1,0.50,0.544010,0.704114,0.613793,445,373,187
2,0.70,0.665541,0.623418,0.643791,394,198,238
3,0.80,0.749482,0.572785,0.649327,362,121,270
4,0.90,0.864615,0.444620,0.587252,281,44,351
5,0.95,0.948571,0.262658,0.411400,166,9,466


## 15. Sample ATE predictions

In [34]:
sample_df = ate_test_df.sample(min(12, len(ate_test_df)), random_state=7)

for _, row in sample_df.iterrows():
    pred = predict_ate_for_tokens(row["tokens"])
    pred_spans = [s for s in pred["spans"] if s["conf"] >= 0.5]

    print("Sentence:", row["sentence"])
    print("Gold:    ", sorted(gold_span_set(row)))
    print("Pred:    ", pred_spans)
    print("Labels:  ", list(zip(row["tokens"], pred["labels"], [round(x, 3) for x in pred["confs"]])))
    print("-" * 100)

Sentence: thoroughly enjoyed this intelligent and quietly atmospheric mystery
Gold:     ['mystery']
Pred:     [{'start': 7, 'end': 7, 'text': 'mystery', 'conf': 0.8620426058769226}]
Labels:   [('thoroughly', 'O', 0.982), ('enjoyed', 'O', 0.989), ('this', 'O', 0.986), ('intelligent', 'O', 0.951), ('and', 'O', 0.982), ('quietly', 'O', 0.987), ('atmospheric', 'O', 0.986), ('mystery', 'B-ASP', 0.862)]
----------------------------------------------------------------------------------------------------
Sentence: good quality coax good quality coax
Gold:     ['coax']
Pred:     [{'start': 1, 'end': 1, 'text': 'quality', 'conf': 0.742499053478241}, {'start': 2, 'end': 2, 'text': 'coax', 'conf': 0.9401965141296387}, {'start': 4, 'end': 4, 'text': 'quality', 'conf': 0.564114511013031}, {'start': 5, 'end': 5, 'text': 'coax', 'conf': 0.9174781441688538}]
Labels:   [('good', 'O', 0.965), ('quality', 'B-ASP', 0.742), ('coax', 'B-ASP', 0.94), ('good', 'O', 0.972), ('quality', 'B-ASP', 0.564), ('coax',